# A tool to keep things in line.

**HIT LINE**:
- We tried the normalizing before to speed up the learning (all weights get gradients in the same way)
- But that **notmalizing was just for the input layer**.
- In the deep net, we have hidden layers -- and we still need to normalize those!!

<img src="./images/choices.png">

## 🚶🏻 Walkthrough and the process

#### **Batch Normalization in a single layer:**

1. **Compute the batch mean:**  
   $$
   \mu = \frac{1}{m} \sum_{i=1}^{m} z^{(i)}
   $$

2. **Compute the batch variance:**  
   $$
   \sigma^2 = \frac{1}{m} \sum_{i=1}^{m} (z^{(i)} - \mu)^2
   $$

3. **Normalize:**  
   $$
   z_{\text{norm}}^{(i)} = \frac{z^{(i)} - \mu}{\sqrt{\sigma^2 + \epsilon}}
   $$

4. **Scale and shift:**  
   $$
   \tilde{z}^{(i)} = \gamma \; z_{\text{norm}}^{(i)} + \beta
   $$

***

- Here, $m$ = batch size  
- $z^{(i)}$ = pre-activation of the $i$th example in the batch  
- $\epsilon$ = small value for stability  
- $\gamma, \beta$ = learnable parameters (per feature/activation)  

### 🤔 Why do we need the learnable parameters `gamma` and `beta`?

- The reason is, **we don't want to strictly force the `z` values** to have mean of 0 and std of 1.
- There **could be something else**.
- And that "something else" can be derived from learning of `gamma` and `beta`.

# The flow

- Inputs feed in to the $l - 1$ layer
- After getting the $z^{[l]}$ we calculate *mean* and *std* of all `z` values, to normalize those.
- And finally shift with $\gamma$ and $\beta$ *(both learnable)*
- And we have $\hat z$ -> Then *activation* and then continue till the last layer.

### Some notes:

- The `bias` term is discarded when the BatchNorm is used.

***

### Additional Notes

- **BatchNorm helps stabilize and accelerate training** by standardizing intermediate pre-activations; it reduces internal covariate shift, making deep networks easier to train.
- **Batch statistics (mean, std) are computed per mini-batch during training**; for inference, running averages collected throughout training are used.
- **If used with convolutional layers**, normalization is done per-channel over all pixels and batch samples.
- **The learnable parameters $\gamma$ and $\beta$ restore the network's representational power** after normalization—they ensure the layer can still learn any mean/variance necessary for each feature.

**Quick formula recap (per feature/channel):**
$$
\hat{z}^{(i)} = \frac{z^{(i)} - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
$$
\tilde{z}^{(i)} = \gamma \hat{z}^{(i)} + \beta
$$

***

### Interview-ready bullet
- The bias is discarded (not needed) because **BatchNorm’s centering step absorbs it, and $\beta$ replaces it as the new learnable shift**.

# 💖 The benefits of BatchNorm

### `1.` Handles **CovarianceShift** like a charm

---

<img src="./images/cov-shift.png">

---

##### **What is Covariate Shift?**

- *Covariate shift* means the **distribution of data changes between training and testing** <u>(for example, training on black cats, but testing on colorful cats).</u>
- Even if the *true* function mapping X to Y hasn’t changed, the network might struggle if the features look very different after changes in data.

##### **Why does this matter inside deep networks?**

- In a deep network, **each hidden layer gets its input from the previous layer**.
- If earlier layers keep changing (as their weights are updated), the hidden units in *later layers* see a moving target--*they’re always seeing **something new***!
- This is just like `hiding` parts of the network: imagine you temporarily cover up the earlier layers (as in Andrew’s notes), the current layer’s only job is to learn a mapping from the values it gets. But as soon as you uncover the earlier layers, those values start changing again.

***

##### **How does BatchNorm help?**

- **BatchNorm standardizes the outputs from every layer to keep the mean and variance stable**.
- This **reduces covariate shift** between layers, so later layers train on more predictable data—even while earlier layers keep updating.

##### **Analogy**

- It’s like every chef in a kitchen being guaranteed that the ingredients they get (`the activations/values`) are always fresh, same quality, and consistently prepared, *no matter how the suppliers upstream change*.
- This lets each layer (chef) focus on its own job, instead of constantly adapting to upstream changes.

***

### **Key Points for Notes**

- **BatchNorm** stabilizes learning, so *later layers* don’t suffer every time an earlier layer gets updated.
- It **weakens the coupling** between layers—each can learn more independently.
- Reduces the internal version of *covariate shift* by fixing the statistical properties (mean, variance) of each hidden unit.
- Makes deep networks much easier to optimize; learning occurs with less back-and-forth between layers.

***

**In summary:**  
BatchNorm is essential because it keeps the flow of information stable across the network, avoiding wild changes and helping every layer learn quickly and efficiently.

### `2.` Does the **Regularization: Regularization and Noisy Mini-batches**


##### **Why does regularization happen?**

- When you use **batch normalization**, the mean and variance for each feature are computed *only on your current mini-batch*—say, 64, 128, or 256 examples.
- Because a mini-batch is just a small sample of your data, the **mean and variance estimates are a little noisy** (not exactly perfect).
- That means when you normalize, the output of the layer gets scaled and shifted using those slightly “off” numbers, so every layer’s activations have a small bump of random noise added.

##### **How is this like dropout?**

- **Dropout** adds noise by randomly zeroing out (or keeping) activations.
- **BatchNorm** adds noise by using slightly different scaling and mean subtraction every batch (since mean and std dev change randomly batch-to-batch).
- The result: *Every forward pass is a little different!*

##### **Why does noise help?**

- **Noise prevents the network from “trusting” any single activation too strongly.**  
  - It forces downstream layers to spread out their “attention” or reliance, and adapt to small unexpected changes.
  - This acts as a **regularization**: the network learns to be less sensitive, which helps it generalize.
- **But...**  
  - The amount of noise is usually **small**—batch norm is not a substitute for dropout!
  - You can use both together: BatchNorm for stability and faster training, Dropout for stronger regularization.

##### **Batch Size Matters**

- **Bigger mini-batch = less noise (mean/variance estimate is more accurate).**
- So, as batch size increases (e.g., 512 examples), the regularization effect of batch norm gets weaker.
- If you want *strong* regularization, use small batch sizes *or* add dropout.

##### **Practical usage**

- **BatchNorm’s main job is to stabilize learning and help the optimizer—its regularization side-effect is minor.**
- Rely on proper regularizers (like dropout, weight decay) if you need strong generalization.

##### **Test Time: A Special Case**

- During training, mean/variance are computed for each mini-batch (with noise).
- At test time (predicting on ONE example), we can’t compute mean/variance from a mini-batch.
- Instead, we use **running averages** (estimated over all training batches) to make sure the predictions make sense.

***

## **Summary in Notes Format**

- **BatchNorm** adds *slight noise* to layer activations (from estimating mean and variance on mini-batches).
- This noise gives a *minor regularizing effect*, similar to but much weaker than dropout.
- **As batch size increases, noise (and regularization effect) decreases.**
- **BatchNorm should NOT be used as your primary regularizer.**
- At prediction time, use running averages of mean/variance (not per-batch estimates).
- **Use BatchNorm for faster, easier optimization—and pair it with regularization methods like Dropout if you want better generalization!**


<img src="./images/bn-regularizatioon.png">